In [16]:
import pandas as pd
import numpy as np
from matplotlib.pyplot import title

### Na końcu tego pliku zapisujemy zawsze aktualne wyczyszczone dane i z nich korzystamy w innych notebookach np first_data_analysis.ipynb

**clean_data.csv** to ramka w której znajduje się wszystkie informacje <br>
**data_coded.csv** już kodujemy niektóre cechy i bardziej format jaki dla modelu

Na razie tylko obserwacja braków danych, jakieś proste operacje na kolumnach.  
**TO DO:** Trzeba będzie każdą kolumnę, która ma wiele wartości po przecinku jakoś inaczej reprezentować (albo usunąć i pozostawić po jednej wartości np gatunku albo jakoś pomyśleć jak to zakodować).
Wizualizacje czasem da sie wykonywać bez tego nawet, ale potem musimy mieć to wyczyszczone i zakodowane na przyszłość do modelów więc trzeba to zrobić

In [ ]:
data = pd.read_csv(('../data/merged/merged_data.csv')) # z notatnika data_merge.ipynb

In [18]:
data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
data['year'] = data['year'].astype(int)

In [19]:
data.isna().sum()

tmdbId                    0
title                     0
budget                    0
revenue                   0
release_date              0
runtime                   0
original_language         0
popularity                0
vote_average              0
vote_count                0
origin_countries        370
spoken_languages        341
year                      0
keywords               3179
genres                   89
director_id              82
director_name            82
director_popularity      82
director_gender          82
writer_id               923
writer_name             923
writer_popularity       923
writer_gender           923
actor4_id               645
actor4_name             645
actor4_popularity       645
actor4_gender           645
actor2_id               256
actor2_name             256
actor2_popularity       256
actor2_gender           256
actor1_id               129
actor1_name             129
actor1_popularity       129
actor1_gender           129
actor5_id           

In [20]:
# zamiana wartości 0 w budżecie na wartości brakujące
data['budget'] = data['budget'].replace(0, np.nan)
data['budget_adjusted'] = data['budget_adjusted'].replace(0, np.nan)

In [21]:
data['budget'].isna().sum()

np.int64(8290)

In [22]:
print(data['revenue_adjusted'].quantile(0.05))
print(data['budget_adjusted'].quantile(0.05))

9913.57
78721.5


In [23]:
# usuwamy filmy o bardzo małym przychodzie (prawdopodobnie błędne dane)
data = data[data['revenue_adjusted'] > 10000]

# nierealnie mały budżet zamieniamy na NaN
data['budget_adjusted'] = data['budget_adjusted'].apply(lambda x: np.nan if x < 10000 else x)
data['budget'] = data['budget'].apply(lambda x: np.nan if x < 10000 else x)
data.head()

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,budget_adjusted,revenue_adjusted
0,5.0,Four Rooms,4000000.0,4257354,1995-12-09,98,en,3.8441,5.892,2810,...,3125.0,Madonna,2.6797,1.0,3130.0,Jennifer Beals,3.9675,1.0,8.449948e+06,8.993604e+06
1,6.0,Judgment Night,21000000.0,12136938,1993-10-15,109,en,2.2315,6.469,368,...,9777.0,Cuba Gooding Jr.,2.5604,2.0,12799.0,Jeremy Piven,2.7546,2.0,4.678756e+07,2.704085e+07
2,11.0,Star Wars,11000000.0,775398007,1977-05-25,121,en,20.6912,8.200,22061,...,12248.0,Alec Guinness,2.0063,2.0,4.0,Carrie Fisher,2.5504,1.0,5.843850e+07,4.119372e+09
3,12.0,Finding Nemo,94000000.0,940335536,2003-05-30,100,en,16.5979,7.817,20364,...,12.0,Alexander Gould,2.0017,2.0,118.0,Geoffrey Rush,2.4578,2.0,1.644709e+08,1.645296e+09
4,13.0,Forrest Gump,55000000.0,677387716,1994-06-23,142,en,26.0309,8.500,29387,...,32.0,Robin Wright,3.9461,1.0,35.0,Sally Field,6.1880,1.0,1.194795e+08,1.471527e+09


In [24]:
# usunięcie filmów o czasie krótszym niż 50min i jednego filmu dłuższego niż 600min
#Bierzemy pod uwagę pełnometrażowe
data = data[(data['runtime'] > 50) & (data['runtime'] < 600)]

In [25]:
data.shape

(19131, 45)

In [26]:
data.isna().sum()

tmdbId                    0
title                     0
budget                 7567
revenue                   0
release_date              0
runtime                   0
original_language         0
popularity                0
vote_average              0
vote_count                0
origin_countries        202
spoken_languages        220
year                      0
keywords               2654
genres                   32
director_id              28
director_name            28
director_popularity      28
director_gender          28
writer_id               665
writer_name             665
writer_popularity       665
writer_gender           665
actor4_id               373
actor4_name             373
actor4_popularity       373
actor4_gender           373
actor2_id               122
actor2_name             122
actor2_popularity       122
actor2_gender           122
actor1_id                70
actor1_name              70
actor1_popularity        70
actor1_gender            70
actor5_id           

In [27]:
# tyle pozostałoby filmów gdyby wyrzucić wszystkie brakujące dane
data.dropna().shape[0]

10033

In [28]:
# dodanie kolumny odpowiadającej za kwartał 
data['quarter'] = data['release_date'].dt.quarter

In [29]:
# dodanie kolumny epoka dzielącej czas na okresy 20-letnie (1950-69, 1970-89, 1990-09, 2010-obecnie)

bins = [1950, 1970, 1990, 2010, 2030] 
labels = ['1950-1969', '1970-1989', '1990-2009', '2010-obecnie']

data['epoka'] = pd.cut(data['year'], bins=bins, labels=labels, right=False)
data['epoka'] = pd.Categorical(data['epoka'], categories=labels, ordered=True)

### Czyszczenie gatunków

In [30]:
# czyszczenie gatunków
def usun_falszywy_gatunek(tekst):
    
    if pd.isna(tekst):
        return tekst
    
    lista_gatunkow = tekst.split(',')

    czyste_gatunki = [g.strip() for g in lista_gatunkow if g.strip() != '(no genres listed)']
    
    if len(czyste_gatunki) == 0:
        return np.nan
    else:
        return ', '.join(czyste_gatunki)

In [31]:
data['genres'] = data['genres'].apply(usun_falszywy_gatunek)
data['genres'] = data['genres'].str.replace('science fiction', 'sci-fi', regex=False)
data['genres'] = data['genres'].str.replace('sciencefiction', 'sci-fi', regex=False)

data['genres'] = data['genres'].str.replace('tv movie', 'tvmovie', regex=False)

data['genres'] = data['genres'].str.replace('music', 'musical', regex=False)
data['genres'] = data['genres'].str.replace('musicalal', 'musical', regex=False)

In [32]:
# dodanie kolumny main_genre

# bo przyda się to do wykresów i staystyk żeby czasem brać po jednym gatunku na film
# jeśli pierwszy gatunek ogólny 'drama' to bierzemy drugi

def wybierz_lepszy_gatunek(genre_string):

    if pd.isna(genre_string):
        return np.nan
    
    gatunki = genre_string.split(',')
    pierwszy_gatunek = gatunki[0].strip()
    
    if pierwszy_gatunek == 'drama' and len(gatunki) > 1:
  
        return gatunki[1].strip()
    else:

        return pierwszy_gatunek

In [33]:
data = data.copy()
data['main_genre'] = data['genres'].apply(wybierz_lepszy_gatunek)

# usunięcie tvmovie bo nie pasują do filmów kinowych
# oraz children bo to też dla tmdb filmy na DVD dla dzieci

data = data[~data['main_genre'].isin(['tvmovie', 'children'])]
data['main_genre'].value_counts()

main_genre
comedy         4728
action         2492
romance        1547
drama          1487
horror         1384
thriller       1145
crime           939
animation       898
adventure       838
documentary     603
family          560
history         525
fantasy         425
sci-fi          420
musical         355
mystery         316
war             277
western         149
Name: count, dtype: int64

In [34]:
# Treaz zliczam wszystkie gatunki występujące (nie tylko main_genre)
wszystkie_gatunki = data['genres'].str.split(',').explode()
wszystkie_gatunki = wszystkie_gatunki.str.strip()

niechciane = ['(no genres listed)', '', None]
wszystkie_gatunki = wszystkie_gatunki[~wszystkie_gatunki.isin(niechciane)]

liczba_unikalnych = wszystkie_gatunki.nunique()
statystyki_gatunkowe = wszystkie_gatunki.value_counts()

statystyki_gatunkowe
# jest ich 22

genres
drama          9863
comedy         7191
thriller       4494
action         4157
romance        3770
adventure      2880
crime          2857
sci-fi         2403
horror         2185
fantasy        1903
family         1857
mystery        1583
animation      1363
history        1017
musical         922
children        736
documentary     715
war             690
western         279
imax            152
tvmovie          30
film-noir        23
Name: count, dtype: int64

In [ ]:
# kodowanie gatunków 0-1

# genres_encoded = data['genres'].str.get_dummies(sep=', ')
# data_coded = pd.concat([data, genres_encoded], axis=1)

In [68]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# lista_gatunkow = list(genres_encoded.columns)
# korelacje = data_coded[lista_gatunkow].corrwith(data_coded['revenue_adjusted']).sort_values(ascending=False)

# plt.figure(figsize=(12, 10))

# sns.barplot(x=korelacje.values, y=korelacje.index, palette='coolwarm')


# plt.title('Korelacja gatunku i przychodu', fontsize = 22, fontweight='bold')
# plt.xlabel('Współczynnik korelacji', fontsize = 16)
# plt.ylabel('Gatunek', fontsize = 1)

# plt.axvline(x=0, color='black', linestyle='-', linewidth=1)

# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)

# plt.show()

### Czyszczenie języki filmów

In [35]:
languages_count = data["original_language"].value_counts()
top_20_languages = languages_count.head(20).index.tolist()
data['original_language'] = data['original_language'].apply(
    lambda x: x if x in top_20_languages else 'other'
)
data['original_language'].value_counts()

original_language
en       11829
other      940
fr         913
ja         803
es         626
hi         502
it         502
zh         472
ru         438
ko         324
de         319
cn         278
ar         230
tr         191
ta         139
ml         134
pl         113
te         106
pt          92
sv          85
no          84
Name: count, dtype: int64

In [72]:
data['origin_countries']

0        United States of America
1        United States of America
2        United States of America
3        United States of America
4        United States of America
                   ...           
20491                      Canada
20495                         NaN
20501                         NaN
20505               Taiwan, Japan
20507                     Denmark
Name: origin_countries, Length: 19120, dtype: object

In [36]:
data['origin_countries'] = data['origin_countries'].str.replace(r'\s+', '', regex=True)
data['origin_countries'] = data['origin_countries'].str.split(',')
data['origin_countries'] = data['origin_countries'].apply(
    lambda x: x if isinstance(x,list) else []
)
df_exploded = data.explode('origin_countries')
df_exploded = df_exploded[df_exploded['origin_countries'] != '']

In [37]:
df_exploded.shape,data.shape

((27230, 48), (19120, 48))

In [38]:
country_counts = df_exploded['origin_countries'].value_counts().reset_index()
country_counts.columns = ['country', 'count']

In [39]:
df_exploded.head(20)

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,budget_adjusted,revenue_adjusted,quarter,epoka,main_genre
0,5.0,Four Rooms,4000000.0,4257354,1995-12-09,98,en,3.8441,5.892,2810,...,1.0,3130.0,Jennifer Beals,3.9675,1.0,8.449948e+06,8.993604e+06,4,1990-2009,comedy
1,6.0,Judgment Night,21000000.0,12136938,1993-10-15,109,en,2.2315,6.469,368,...,2.0,12799.0,Jeremy Piven,2.7546,2.0,4.678756e+07,2.704085e+07,4,1990-2009,action
2,11.0,Star Wars,11000000.0,775398007,1977-05-25,121,en,20.6912,8.200,22061,...,2.0,4.0,Carrie Fisher,2.5504,1.0,5.843850e+07,4.119372e+09,2,1970-1989,adventure
3,12.0,Finding Nemo,94000000.0,940335536,2003-05-30,100,en,16.5979,7.817,20364,...,2.0,118.0,Geoffrey Rush,2.4578,2.0,1.644709e+08,1.645296e+09,2,1990-2009,animation
4,13.0,Forrest Gump,55000000.0,677387716,1994-06-23,142,en,26.0309,8.500,29387,...,1.0,35.0,Sally Field,6.1880,1.0,1.194795e+08,1.471527e+09,2,1990-2009,comedy
5,14.0,American Beauty,15000000.0,356296601,1999-09-15,122,en,9.8464,8.000,12880,...,1.0,516.0,Annette Bening,3.7232,1.0,2.898646e+07,6.885186e+08,3,1990-2009,romance
6,16.0,Dancer in the Dark,12500000.0,40061153,2000-09-01,140,en,3.5533,7.850,1961,...,1.0,52.0,David Morse,2.9324,2.0,2.336985e+07,7.489784e+07,3,1990-2009,crime
6,16.0,Dancer in the Dark,12500000.0,40061153,2000-09-01,140,en,3.5533,7.850,1961,...,1.0,52.0,David Morse,2.9324,2.0,2.336985e+07,7.489784e+07,3,1990-2009,crime
6,16.0,Dancer in the Dark,12500000.0,40061153,2000-09-01,140,en,3.5533,7.850,1961,...,1.0,52.0,David Morse,2.9324,2.0,2.336985e+07,7.489784e+07,3,1990-2009,crime
6,16.0,Dancer in the Dark,12500000.0,40061153,2000-09-01,140,en,3.5533,7.850,1961,...,1.0,52.0,David Morse,2.9324,2.0,2.336985e+07,7.489784e+07,3,1990-2009,crime


In [40]:
country_counts

,country,count
0,UnitedStatesofAmerica,10519
1,UnitedKingdom,2340
2,France,1890
3,Germany,1149
4,India,1047
...,...,...
136,Madagascar,1
137,Andorra,1
138,Mauritius,1
139,St.KittsandNevis,1


In [41]:
top5=country_counts[1:6]['country'].tolist()

In [42]:

# # Stwórz nową kolumnę
data['one_or_more_origin'] = data['origin_countries'].apply(
    lambda x: 1 if len(x) == 1 else (0 if len(x) > 1 else '')
)
data['origin_in_US'] = data['origin_countries'].apply(
    lambda x: 1 if "UnitedStatesofAmerica" in x else 0
)
def in_top_5(x):
    for i in x:
        if i in top5:
            return True
    return False

data['origin_in_top_5']=data['origin_countries'].apply(
    lambda x: 1 if  in_top_5(x) else 0
)
data[['origin_countries','origin_in_US','one_or_more_origin','origin_in_top_5']]

,origin_countries,origin_in_US,one_or_more_origin,origin_in_top_5
0,[UnitedStatesofAmerica],1,1,0
1,[UnitedStatesofAmerica],1,1,0
2,[UnitedStatesofAmerica],1,1,0
3,[UnitedStatesofAmerica],1,1,0
4,[UnitedStatesofAmerica],1,1,0
...,...,...,...,...
20491,[Canada],0,1,0
20495,[],0,,0
20501,[],0,,0
20505,"[Taiwan, Japan]",0,0,1


In [43]:
type(data['origin_countries'][0])


list

In [44]:
data.columns[:20]

Index(['tmdbId', 'title', 'budget', 'revenue', 'release_date', 'runtime',
       'original_language', 'popularity', 'vote_average', 'vote_count',
       'origin_countries', 'spoken_languages', 'year', 'keywords', 'genres',
       'director_id', 'director_name', 'director_popularity',
       'director_gender', 'writer_id'],
      dtype='object')

In [45]:
lista_gatunkow = list(set(data['main_genre'].tolist()))
lista_gatunkow.remove(np.nan)
lista_gatunkow

['sci-fi',
 'thriller',
 'musical',
 'family',
 'comedy',
 'drama',
 'fantasy',
 'romance',
 'war',
 'history',
 'adventure',
 'mystery',
 'documentary',
 'animation',
 'action',
 'horror',
 'crime',
 'western']

### Zapis ramki, która była używana w first_data_cleaning_analysis

In [46]:
data.to_csv("../data/merged/clean_data.csv", index = False)